# 02 · Exploración y limpieza de datos

**Objetivo:** entender la calidad de los datos, corregir valores imposibles y ver qué variables se relacionan con el incumplimiento, **usando solo la muestra de entrenamiento**.

El notebook tiene tres partes:
- **A. Calidad de datos:** faltantes, valores imposibles y valores extremos.
- **B. Limpieza:** reglas de negocio que se aprenden con entrenamiento y se aplican igual a prueba.
- **C. Relación con la variable objetivo:** tasa de malos por tramos de cada variable. Es la antesala del WoE de la etapa 3.

> **Regla de oro:** la muestra de prueba no se explora. Solo se le aplican, al final, las mismas reglas aprendidas con entrenamiento.

## Entorno

Conecta Google Drive, clona el repositorio e instala las librerías con las versiones fijadas en `requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # Donde está cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.datos import copiar_datos_crudos, obtener_particiones, resumen_estructura
from src.eda import (
    graficar_tasa_malos,
    pares_correlacionados,
    resumen_percentiles,
    tasa_malos,
    tasa_malos_por_faltante,
    tasa_malos_por_tramos,
)
from src.limpieza import ATRASOS_MAXIMO_POSIBLE, VARIABLES_ATRASO, LimpiadorCredito

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)

In [ ]:
# En una sesión nueva de Colab se copian los datos desde Drive y se
# regenera la partición del notebook 01 (idéntica gracias a la semilla).
copiar_datos_crudos(CARPETA_DRIVE)
train, test = obtener_particiones()

Y = config.COLUMNA_OBJETIVO
TASA_GLOBAL = train[Y].mean()
print(f"Entrenamiento: {len(train):,} clientes | Tasa de malos: {TASA_GLOBAL:.2%}")

# A. Calidad de datos

## A.1 Estadísticos con percentiles extremos

La media es engañosa cuando hay valores atípicos: un solo cliente con un valor absurdo la mueve mucho. Por eso se revisan los **percentiles 1 % y 99 %** junto al mínimo y el máximo. Si el máximo está muy lejos del percentil 99, hay unos pocos valores extremos.

In [ ]:
resumen_percentiles(train.drop(columns=Y))

## A.2 Registros duplicados

Se cuentan filas con valores idénticos en todas las columnas. Como cada fila tiene un `id_cliente` distinto, un duplicado puede ser un error de carga o dos clientes con el mismo perfil, algo plausible en variables con muchos ceros (sin atrasos, sin dependientes, etc.).

In [ ]:
duplicados = train.duplicated().sum()
print(f"Filas duplicadas en entrenamiento: {duplicados:,} ({duplicados / len(train):.2%})")

## A.3 Faltantes: ¿son informativos?

Antes de decidir qué hacer con los faltantes conviene preguntarse: **¿los clientes sin dato se comportan distinto?** Si su tasa de malos es diferente a la de los clientes con dato, que falte el dato **es información** y no conviene "rellenarlo" con la mediana, porque se borraría esa señal.

In [ ]:
tasa_malos_por_faltante(train, ["ingreso_mensual", "num_dependientes"])

## A.4 Edades imposibles

Para contratar un crédito se necesita ser mayor de edad. Una edad menor a 18 es un error de captura.

In [ ]:
print(f"Clientes con edad < 18: {(train['edad'] < 18).sum()}")
train["edad"].value_counts().sort_index().head(5)

## A.5 Códigos especiales en las variables de atrasos

Las tres variables cuentan **cuántas veces** se atrasó el cliente en los últimos 2 años. En 2 años hay a lo más 24 pagos mensuales, así que **un valor mayor a 24 es imposible** como conteo: probablemente es un código del sistema de origen (por ejemplo, "sin información").

Se revisa: qué valores aparecen arriba de 24, si los códigos aparecen juntos en las tres variables y qué tasa de malos tienen esos clientes.

In [ ]:
for var in VARIABLES_ATRASO:
    conteo = train.loc[train[var] > ATRASOS_MAXIMO_POSIBLE, var].value_counts().sort_index()
    print(f"{var}: {conteo.to_dict()}")

In [ ]:
# ¿En cuántas de las tres variables aparece el código, por cliente?
num_codigos = (train[VARIABLES_ATRASO] > ATRASOS_MAXIMO_POSIBLE).sum(axis=1)
print("Clientes según número de variables con código:")
print(num_codigos.value_counts().sort_index().to_string())

In [ ]:
# Tasa de malos: clientes con código vs. sin código
tiene_codigo = (num_codigos > 0).map({True: "con código", False: "sin código"})
tasa_malos(train, tiene_codigo)

## A.6 Uso de crédito revolvente

Es saldo entre límite. Lo normal es que esté entre 0 y 1 (0 % a 100 % del límite). Un valor un poco arriba de 1 es posible (sobregiro), pero valores de miles no tienen sentido como proporción.

In [ ]:
rangos_uso = pd.cut(train["uso_credito_revolvente"], bins=[-np.inf, 1, 2, 10, np.inf],
                    labels=["0 a 1", "1 a 2", "2 a 10", "más de 10"])
tasa_malos(train, rangos_uso)

## A.7 Razón de deuda cuando no hay ingreso

La razón de deuda es **pagos de deuda ÷ ingreso**. Si el ingreso falta o es cero, la división no está definida, así que el valor guardado no puede ser una razón. Se compara la distribución de `razon_deuda` según el ingreso.

In [ ]:
grupo_ingreso = np.select(
    [train["ingreso_mensual"].isna(), train["ingreso_mensual"] == 0],
    ["ingreso faltante", "ingreso = 0"],
    default="ingreso > 0",
)
train.groupby(grupo_ingreso)["razon_deuda"].describe(percentiles=[0.5, 0.75, 0.99])

## Hallazgos de la parte A

- **Duplicados:** 351 filas (0.33 %). Se conservan: cada una tiene un `id_cliente` distinto y perfiles idénticos son plausibles en variables con muchos ceros.
- **Faltantes informativos:** los clientes sin ingreso tienen una tasa de malos de 5.48 % frente a 6.98 % con dato; sin número de dependientes, 4.15 % frente a 6.75 %. Imputar con la mediana borraría esa diferencia, así que el faltante se conserva como grupo propio.
- **Edad:** en entrenamiento no hay edades menores a 18 (el mínimo es 21). La regla se mantiene porque también se aplica a prueba y a clientes futuros.
- **Códigos 96 y 98 en atrasos:** 182 clientes, y en todos los casos el código aparece en las tres variables a la vez. Su tasa de malos es de 54.40 %, frente a 6.60 % del resto. Son clientes de muy alto riesgo: eliminarlos o confundirlos con "cero atrasos" perdería esa señal.
- **Uso de crédito revolvente:** entre 1 y 2 veces el límite, la tasa de malos es de 39.62 % (sobregiro real, alto riesgo). Por arriba de 10 baja a 9.47 %, cerca del promedio: esos valores no se comportan como sobregiros y se tratan como errores de captura. Si solo se acotaran, esos clientes caerían en el grupo de mayor riesgo sin serlo.
- **Razón de deuda sin ingreso:** con ingreso positivo, la mediana es 0.29; con ingreso cero o faltante, la mediana pasa de 900. Sin ingreso, el valor no es una razón (probablemente es el monto de la deuda), así que se trata como faltante.

# B. Limpieza

## B.1 Reglas

| Regla | Por qué |
|---|---|
| Edad < 18 → faltante | Valor imposible para un acreditado. |
| Atrasos > 24 → faltante | No es un conteo posible en 2 años; es un código especial. |
| Razón de deuda sin ingreso → faltante | Sin ingreso no existe la razón deuda/ingreso. |
| Uso de crédito > 10 → faltante | No se comporta como un sobregiro real (ver A.6): es un error de captura. |
| Uso de crédito, razón de deuda e ingreso acotados en el percentil 99 | Limita el peso de valores extremos (*winsorización*). |

**¿Por qué no se eliminan esos clientes?** Porque en producción también llegarán solicitudes con datos raros y el modelo debe calificarlas. Además, eliminar filas puede sesgar la muestra si esos clientes tienen un riesgo distinto.

**¿Por qué no se imputan los faltantes?** En un scorecard, los faltantes forman **su propio grupo** al calcular el WoE (etapa 3). Así el modelo aprende el riesgo de "no tener el dato" en lugar de esconderlo con un valor inventado. XGBoost y LightGBM también manejan faltantes directamente.

**¿Por qué el tope se calcula solo con entrenamiento?** Por la misma razón que se separó la muestra de prueba: el percentil 99 es un parámetro aprendido de los datos. Si se calculara con todos los datos, la muestra de prueba influiría en el modelo (fuga de información).

In [ ]:
limpiador = LimpiadorCredito(percentil_tope=0.99).fit(train)

print("Topes aprendidos con entrenamiento (percentil 99):")
print(limpiador.topes_.to_string(float_format="{:,.4f}".format))

In [ ]:
# Registros que modifica cada regla en entrenamiento
limpiador.reporte(train)

In [ ]:
train_limpio = limpiador.transform(train)
test_limpio = limpiador.transform(test)   # Mismas reglas y mismos topes

resumen_estructura(train_limpio)

In [ ]:
train_limpio.to_csv(config.ARCHIVO_TRAIN_LIMPIO)
test_limpio.to_csv(config.ARCHIVO_TEST_LIMPIO)
print("Guardado:", config.ARCHIVO_TRAIN_LIMPIO.relative_to(config.RAIZ), "y",
      config.ARCHIVO_TEST_LIMPIO.relative_to(config.RAIZ))

# C. Relación con la variable objetivo

Para cada variable se divide a los clientes en tramos y se calcula la **tasa de malos** de cada tramo. La línea punteada es la tasa global.

Qué buscar:
- **Tendencia clara** (la tasa sube o baja de forma ordenada): la variable separa bien a buenos de malos y será buena predictora.
- **Tasa plana**, igual en todos los tramos: la variable aporta poco.
- **Grupo "Faltante"**: si su tasa difiere de la global, el faltante es informativo.

Este análisis es la base intuitiva del **WoE** y el **IV** de la etapa 3, que miden lo mismo de forma numérica.

- Variables continuas: se cortan en **deciles** (10 grupos con el mismo número de clientes).
- Variables de conteo: un grupo por valor (0, 1, 2, ...), con el último como "k o más".

In [ ]:
# Variable -> tope para variables de conteo (None = deciles)
tramos_por_variable = {
    "uso_credito_revolvente": None,
    "edad": None,
    "razon_deuda": None,
    "ingreso_mensual": None,
    "num_creditos_abiertos": None,
    "atrasos_30_59": 3,
    "atrasos_60_89": 3,
    "atrasos_90": 3,
    "num_creditos_inmobiliarios": 4,
    "num_dependientes": 4,
}

tablas_tramos = {}
for variable, tope in tramos_por_variable.items():
    tabla = tasa_malos_por_tramos(train_limpio, variable, tope=tope)
    tablas_tramos[variable] = tabla
    graficar_tasa_malos(tabla, f"Tasa de malos por tramo de {variable}", TASA_GLOBAL)
    plt.show()

## C.2 Variables muy correlacionadas entre sí

Si dos variables dicen casi lo mismo (alta correlación), meter ambas a una regresión logística puede volver inestables sus coeficientes (**multicolinealidad**). Aquí solo se identifican los pares; la decisión de cuál conservar se toma en la etapa 3 con el IV.

In [ ]:
pares_correlacionados(train_limpio.drop(columns=Y), umbral=0.5)

## Resumen de la etapa

- Se revisó la calidad de datos **solo con la muestra de entrenamiento**.
- Reglas aplicadas: edad < 18, atrasos > 24, razón de deuda sin ingreso y uso de crédito > 10 pasan a faltante; uso de crédito, razón de deuda e ingreso se acotan en el percentil 99 de entrenamiento.
- Los faltantes se conservan como grupo propio para la etapa 3.
- Se guardaron `train_limpio.csv` y `test_limpio.csv` en `data/processed/`.

**Siguiente paso:** notebook `03_woe_iv.ipynb`, agrupación de variables y cálculo de WoE e IV.